# Stage 6-2: Multiclass 실험 비교

이 노트북은 `experiments/run_all.py`로 실행한 multiclass task의 MLP와 CNN 학습 결과를 비교한다.

**실험 설정**
| 모델 | exp_name |
|---|---|
| MLP | `multiclass_mlp_ep10_lr0.01_bs64` |
| CNN | `multiclass_cnn_ep10_lr0.001_bs32` |

**주의**: `outputs/` 파일이 없으면 아래 셀에서 `run_all.py`를 실행한다.

In [ ]:
# sys.path 설정 — 프로젝트 루트를 모듈 검색 경로에 추가한다.
import sys, os
sys.path.insert(0, os.path.abspath("../.."))

In [ ]:
import subprocess
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image

In [ ]:
from src.data.mnist import MNISTDataset, get_task_spec
from src.data.dataloader import Dataloader
from src.models.mlp import MLP
from src.core.evaluator import Evaluator
from src.core.predictor import Predictor
from src.core.visualizer import Visualizer
from src.utils import checkpoints

In [ ]:
# 학습 결과 없으면 run_all.py 실행
TASK = "multiclass"
MLP_DIR = "../../outputs/multiclass_mlp_ep10_lr0.01_bs64"
CNN_DIR = "../../outputs/multiclass_cnn_ep10_lr0.001_bs32"
DATASET_DIR = "/mnt/d/datasets/mnist"

if not os.path.exists(f"{MLP_DIR}/model.npz"):
    print("학습 결과 없음 — run_all.py 실행 중... (수 분 소요)")
    r = subprocess.run(
        [sys.executable, "../../experiments/run_all.py"],
        capture_output=True, text=True,
    )
    print(r.stdout[-1000:])
else:
    print("학습 결과 확인:")
    print(f"  MLP: {MLP_DIR}/model.npz")
    cnn_status = "있음" if os.path.exists(f"{CNN_DIR}/model.npz") else "없음 (CuPy 환경 필요)"
    print(f"  CNN: {cnn_status}")

## 1. Multiclass 개요

MNIST 0~9 숫자 10개 클래스를 분류하는 task이다.

| 항목 | 내용 |
|---|---|
| 출력 차원 | 10 (클래스 수) |
| loss | cross-entropy (softmax 내장) |
| metric | accuracy (argmax 기반) |
| 후처리 | argmax — 가장 큰 logit 인덱스 |

In [ ]:
# task_spec 확인
ts = get_task_spec(TASK)
print("task_spec:", ts)

test_ds = MNISTDataset("test", TASK, dataset_dir=DATASET_DIR)
print(f"test set: {len(test_ds)} samples, output_dim={ts['output_dim']}")

## 2. MLP 결과 — 학습 곡선과 예측 grid

In [ ]:
# MLP 결과 파일 확인
mlp_log = f"{MLP_DIR}/training_log.png"
mlp_pred = f"{MLP_DIR}/predictions.png"

for path, label in [(mlp_log, "training_log.png"), (mlp_pred, "predictions.png")]:
    status = "있음" if os.path.exists(path) else "없음"
    print(f"  MLP {label}: {status}")

In [ ]:
# MLP 결과 시각화 (파일이 있는 경우)
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

if os.path.exists(mlp_log):
    axes[0].imshow(Image.open(mlp_log))
    axes[0].set_title("MLP: Training Log")
else:
    axes[0].text(0.5, 0.5, "training_log.png 없음", ha="center", va="center")
    axes[0].set_title("MLP: Training Log (없음)")
axes[0].axis("off")

if os.path.exists(mlp_pred):
    axes[1].imshow(Image.open(mlp_pred))
    axes[1].set_title("MLP: Predictions Grid")
else:
    axes[1].text(0.5, 0.5, "predictions.png 없음", ha="center", va="center")
    axes[1].set_title("MLP: Predictions Grid (없음)")
axes[1].axis("off")

plt.suptitle("Multiclass MLP (10 epochs, lr=0.01, bs=64)", fontsize=13)
plt.tight_layout()
plt.show()

## 3. MLP checkpoint 로드 후 직접 평가

In [ ]:
# MLP 모델 로드 및 재평가
test_loader = Dataloader(test_ds, batch_size=256, shuffle=False)

mlp_model = MLP(task=TASK, seed=42)
if os.path.exists(f"{MLP_DIR}/model.npz"):
    checkpoints.load(mlp_model, f"{MLP_DIR}/model.npz")
    print("MLP checkpoint 로드 완료")

evaluator_mlp = Evaluator(mlp_model, ts)
result_mlp = evaluator_mlp.evaluate(test_loader)
print(f"MLP test: loss={result_mlp['loss']:.4f}, accuracy={result_mlp['metric']:.4f}, samples={result_mlp['num_samples']}")

In [ ]:
# MLP 예측 grid 직접 생성
import tempfile
tmpdir = tempfile.mkdtemp()

images_sample = test_ds.images[:32]
labels_sample = test_ds.labels_raw[:32]

predictor_mlp = Predictor(mlp_model, ts)
pred_mlp = predictor_mlp.predict(images_sample)

vis = Visualizer(output_dir=tmpdir)
grid_path = vis.plot_predictions(
    images=images_sample,
    labels=labels_sample,
    predictions=pred_mlp["predictions"],
    filename="mlp_preds.png",
    n=16,
)

correct = (pred_mlp["predictions"] == labels_sample).sum()
print(f"32개 샘플 정확도: {correct}/32 = {correct/32:.3f}")

plt.figure(figsize=(14, 4))
plt.imshow(Image.open(grid_path))
plt.axis("off")
plt.title("MLP 예측 grid (32개 test 샘플)")
plt.tight_layout()
plt.show()

## 4. CNN 결과 비교 (CuPy 환경 실행 필요)

In [ ]:
# CNN 결과 파일 확인
cnn_log = f"{CNN_DIR}/training_log.png"
cnn_pred = f"{CNN_DIR}/predictions.png"

for path, label in [(cnn_log, "training_log.png"), (cnn_pred, "predictions.png")]:
    status = "있음" if os.path.exists(path) else "없음 (cupy_py311_cuda118 환경에서 실행 필요)"
    print(f"  CNN {label}: {status}")

In [ ]:
# MLP vs CNN 학습 곡선 비교 (CNN 결과가 있는 경우)
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, img_path, title in [
    (axes[0], mlp_log, "MLP (lr=0.01, bs=64)"),
    (axes[1], cnn_log, "CNN (lr=0.001, bs=32)"),
]:
    if os.path.exists(img_path):
        ax.imshow(Image.open(img_path))
    else:
        ax.text(0.5, 0.5, "결과 없음", ha="center", va="center", fontsize=14)
    ax.set_title(f"Multiclass {title}")
    ax.axis("off")

plt.suptitle("Multiclass: MLP vs CNN 학습 곡선 비교", fontsize=13)
plt.tight_layout()
plt.show()

In [ ]:
# MLP vs CNN 예측 grid 비교
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, img_path, title in [
    (axes[0], mlp_pred, "MLP Predictions"),
    (axes[1], cnn_pred, "CNN Predictions"),
]:
    if os.path.exists(img_path):
        ax.imshow(Image.open(img_path))
    else:
        ax.text(0.5, 0.5, "결과 없음", ha="center", va="center", fontsize=14)
    ax.set_title(f"Multiclass {title}")
    ax.axis("off")

plt.suptitle("Multiclass: MLP vs CNN 예측 grid 비교", fontsize=13)
plt.tight_layout()
plt.show()

## 5. 요약

**Multiclass task**
- 10개 클래스 분류, metric = accuracy
- 예측 후처리: argmax (softmax 없이 logit에 직접 적용 가능)
- MLP(784→256→128→10)는 SGD lr=0.01로 충분한 수렴을 달성한다
- CNN은 합성곱 레이어로 공간 특징을 추출하여 더 높은 정확도를 달성한다

| 모델 | 파라미터 수 | test accuracy (참고) |
|---|---|---|
| MLP | 235,146 | ~0.96 |
| CNN | 824,458 | ~0.99 |